In [0]:
# Shared parsing library for the updated blob pipeline.
# One consolidated, exact-version dependency installation replaces the original
# duplicated floating-version installs. Job-scoped libraries may replace this cell.

%pip install ftfy==6.3.1 mammoth==1.12.0 Resiliparse==1.0.8 pypdfium2==5.11.0 loky==3.5.6 striprtf==0.0.32
%pip install pytesseract==0.3.13 Pillow==12.3.0 PyMuPDF==1.28.0 openpyxl==3.1.5 defusedxml==0.7.1 python-magic==0.4.27 olefile==0.47 docx2txt==0.9 beautifulsoup4==4.15.0
dbutils.library.restartPython()

In [0]:
%run ./blob_decoder

In [0]:
import io
import json
import re
import shutil
import subprocess
import zipfile
from concurrent.futures import TimeoutError as FutureTimeout
from importlib import metadata as importlib_metadata
from typing import Optional, Tuple


def _optional_import(module_name, attribute=None):
    try:
        module = __import__(module_name, fromlist=[attribute] if attribute else ["*"])
        return getattr(module, attribute) if attribute else module
    except Exception:
        return None


try:
    import ftfy as _ftfy
    from ftfy import TextFixerConfig as _FtfyConfig
except Exception as exc:
    raise RuntimeError(
        "ftfy is required. Attach a pinned dependency environment to the Databricks job."
    ) from exc

try:
    from loky import ProcessPoolExecutor
except Exception as exc:
    raise RuntimeError(
        "loky is required. Attach a pinned dependency environment to the Databricks job."
    ) from exc


_magic = _optional_import("magic")
_olefile = _optional_import("olefile")
_pdfium = _optional_import("pypdfium2")
_fitz = _optional_import("fitz")
_pytess = _optional_import("pytesseract")
_PILImage = _optional_import("PIL.Image")
_mammoth = _optional_import("mammoth")
_docx2txt = _optional_import("docx2txt")
_openpyxl_load = _optional_import("openpyxl", "load_workbook")
_rtf_to_text = _optional_import("striprtf.striprtf", "rtf_to_text")
_RespHTMLTree = _optional_import("resiliparse.parse.html", "HTMLTree")
_resp_extract = _optional_import("resiliparse.extract.html2text", "extract_plain_text")


def dependency_report():
    packages = [
        "ftfy", "loky", "mammoth", "resiliparse", "pypdfium2",
        "PyMuPDF", "pytesseract", "Pillow", "striprtf", "openpyxl",
        "python-magic", "olefile", "defusedxml",
    ]
    result = {}
    for package in packages:
        try:
            result[package] = importlib_metadata.version(package)
        except Exception:
            result[package] = None
    result["tesseract_binary"] = shutil.which("tesseract")
    return result


# ==================== MIME DETECTION ====================

def _classify_zip(content: bytes) -> str:
    try:
        with zipfile.ZipFile(io.BytesIO(content)) as archive:
            names = {name.lower() for name in archive.namelist()}
        if any(name.startswith("word/") for name in names):
            return "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
        if any(name.startswith("xl/") for name in names):
            return "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
        if any(name.startswith("ppt/") for name in names):
            return "application/vnd.openxmlformats-officedocument.presentationml.presentation"
        return "application/zip"
    except Exception:
        return "application/zip"


def _classify_ole(content: bytes) -> str:
    if _olefile is None:
        return "application/x-ole-storage"
    try:
        with _olefile.OleFileIO(io.BytesIO(content)) as ole:
            if ole.exists("WordDocument"):
                return "application/msword"
            if ole.exists("Workbook") or ole.exists("Book"):
                return "application/vnd.ms-excel"
            if ole.exists("PowerPoint Document"):
                return "application/vnd.ms-powerpoint"
    except Exception:
        pass
    return "application/x-ole-storage"


def detect_mime(content: bytes) -> str:
    if not content:
        return "application/octet-stream"
    if content.startswith(PDF_MAGIC):
        return "application/pdf"
    if content.startswith(RTF_MAGIC) or content.startswith(RTF_MAGIC_LOOSE):
        return "text/rtf"
    if content.startswith(ZIP_MAGIC):
        return _classify_zip(content)
    if content.startswith(OLE_MAGIC):
        return _classify_ole(content)
    if content.startswith(JPEG_MAGIC):
        return "image/jpeg"
    if content.startswith(PNG_MAGIC):
        return "image/png"
    if content.startswith(TIFF_LE_MAGIC) or content.startswith(TIFF_BE_MAGIC):
        return "image/tiff"
    if content.startswith(GIF87_MAGIC) or content.startswith(GIF89_MAGIC):
        return "image/gif"
    if _magic is not None:
        try:
            return _magic.Magic(mime=True).from_buffer(content) or "application/octet-stream"
        except Exception:
            pass
    return "application/octet-stream"


_TEXT_APPLICATION_TYPES = {
    "application/json", "application/xml", "application/xhtml+xml",
    "application/rtf", "application/sql", "application/csv",
}


def is_binary_content_type(content_type: str) -> bool:
    value = (content_type or "application/octet-stream").lower()
    if value.startswith(("image/", "audio/", "video/", "font/")):
        return True
    if value.startswith("text/") or value in _TEXT_APPLICATION_TYPES:
        return False
    if value == "application/octet-stream":
        return False  # eligible for guarded text detection
    # Supported structured formats are handled before this predicate.
    return value.startswith("application/")


# ==================== FORMAT EXTRACTORS ====================

def _extract_pdf_pdfium(content: bytes) -> Optional[str]:
    if _pdfium is None:
        return None
    pdf = None
    try:
        pdf = _pdfium.PdfDocument(io.BytesIO(content))
        parts = []
        for index in range(len(pdf)):
            page = pdf[index]
            text_page = None
            try:
                text_page = page.get_textpage()
                value = text_page.get_text_range() or ""
                if value.strip():
                    parts.append(value)
            finally:
                if text_page is not None:
                    try:
                        text_page.close()
                    except Exception:
                        pass
                try:
                    page.close()
                except Exception:
                    pass
        return "\n".join(parts) if parts else None
    except Exception:
        return None
    finally:
        if pdf is not None:
            try:
                pdf.close()
            except Exception:
                pass


def _extract_pdf_pymupdf(content: bytes) -> Optional[str]:
    if _fitz is None:
        return None
    document = None
    try:
        document = _fitz.open(stream=content, filetype="pdf")
        if document.needs_pass:
            try:
                document.authenticate("")
            except Exception:
                pass
        parts = []
        for page in document:
            value = page.get_text("text") or ""
            if value.strip():
                parts.append(value)
        return "\n".join(parts) if parts else None
    except Exception:
        return None
    finally:
        if document is not None:
            try:
                document.close()
            except Exception:
                pass


def _extract_pdf_ocr(content: bytes) -> Optional[str]:
    if _fitz is None or _pytess is None or _PILImage is None or not shutil.which("tesseract"):
        return None
    if len(content) > OCR_MAX_PDF_SIZE_MB * 1024 * 1024:
        return None
    document = None
    try:
        document = _fitz.open(stream=content, filetype="pdf")
        parts = []
        for index, page in enumerate(document):
            if index >= OCR_MAX_PAGES:
                break
            image = None
            pixmap = None
            try:
                pixmap = page.get_pixmap(matrix=_fitz.Matrix(150 / 72, 150 / 72))
                image = _PILImage.open(io.BytesIO(pixmap.tobytes("png")))
                value = _pytess.image_to_string(
                    image, lang=OCR_LANG, timeout=OCR_PAGE_TIMEOUT_SEC
                )
                if value and value.strip():
                    parts.append(value.strip())
            except Exception:
                continue
            finally:
                if image is not None:
                    try:
                        image.close()
                    except Exception:
                        pass
                pixmap = None
        return "\n\n".join(parts) if parts else None
    except Exception:
        return None
    finally:
        if document is not None:
            try:
                document.close()
            except Exception:
                pass


def _extract_docx(content: bytes) -> Optional[str]:
    if _mammoth is not None:
        try:
            result = _mammoth.convert_to_markdown(io.BytesIO(content))
            if result.value and result.value.strip():
                return result.value
        except Exception:
            pass
    if _docx2txt is not None:
        try:
            value = _docx2txt.process(io.BytesIO(content))
            return value if value and value.strip() else None
        except Exception:
            pass
    return None


def _extract_excel(content: bytes) -> Optional[str]:
    if _openpyxl_load is None:
        return None
    workbook = None
    try:
        workbook = _openpyxl_load(io.BytesIO(content), read_only=True, data_only=True)
        parts = []
        for sheet_name in workbook.sheetnames:
            sheet = workbook[sheet_name]
            for row in sheet.iter_rows(values_only=True):
                value = " ".join(str(cell) for cell in row if cell is not None)
                if value.strip():
                    parts.append(value)
        return "\n".join(parts) if parts else None
    except Exception:
        return None
    finally:
        if workbook is not None:
            try:
                workbook.close()
            except Exception:
                pass


def _extract_html(content: bytes) -> Optional[str]:
    for encoding in ("utf-8", "windows-1252", "latin-1"):
        try:
            html = content.decode(encoding, errors="strict")
        except UnicodeDecodeError:
            continue
        if "<" not in html or ">" not in html:
            continue
        if _RespHTMLTree is not None and _resp_extract is not None:
            try:
                tree = _RespHTMLTree.parse(html)
                value = _resp_extract(tree, preserve_formatting=False, main_content=False)
                if value and value.strip():
                    return value
            except Exception:
                pass
        try:
            from bs4 import BeautifulSoup
            soup = BeautifulSoup(html, "html.parser")
            for element in soup(["script", "style", "head", "title", "meta"]):
                element.decompose()
            value = soup.get_text(separator=" ", strip=True)
            if value:
                return value
        except Exception:
            pass
    return None


def _extract_rtf(content: bytes) -> Optional[str]:
    if _rtf_to_text is None:
        return None
    for encoding in ("cp1252", "utf-8"):
        try:
            source = content.decode(encoding, errors="strict")
        except UnicodeDecodeError:
            continue
        if not (source.startswith("{\\rtf") or source.startswith("{\\")):
            continue
        try:
            value = _rtf_to_text(source, errors="ignore")
            return value if value and value.strip() else None
        except Exception:
            return None
    return None


# ==================== GUARDED TEXT DETECTION ====================

def _text_is_plausible(value: str) -> bool:
    if not value:
        return False
    sample = value[:100_000]
    if "\x00" in sample:
        return False
    controls = sum(1 for ch in sample if ord(ch) < 32 and ch not in "\n\r\t\f")
    printable = sum(1 for ch in sample if ch.isprintable() or ch in "\n\r\t\f")
    return controls / len(sample) <= 0.01 and printable / len(sample) >= 0.90


def _guess_text(content: bytes) -> Tuple[Optional[str], Optional[str]]:
    if not content:
        return None, None
    candidates = []
    if content.startswith(b"\xff\xfe"):
        candidates.append(("utf-16-le", content))
    elif content.startswith(b"\xfe\xff"):
        candidates.append(("utf-16-be", content))
    elif content.startswith(b"\xef\xbb\xbf"):
        candidates.append(("utf-8-sig", content))
    candidates.extend((encoding, content) for encoding in ("utf-8", "cp1252", "latin-1"))
    for encoding, value in candidates:
        try:
            decoded = value.decode(encoding, errors="strict")
        except UnicodeDecodeError:
            continue
        if _text_is_plausible(decoded):
            return decoded, encoding
    return None, None


STATUS_LABELS = {
    "decoded": "Decoded",
    "binary_content": "Binary content - not text",
    "binary_unknown": "Binary data - unable to decode safely",
    "pdf_extraction_failed": "PDF extraction failed",
    "docx_extraction_failed": "DOCX extraction failed",
    "excel_extraction_failed": "Excel extraction failed",
    "html_extraction_failed": "HTML extraction failed",
    "rtf_extraction_failed": "RTF extraction failed",
    "unsupported_powerpoint": "PowerPoint content - unsupported",
    "unsupported_binary": "Unsupported binary content",
    "failed_to_decode": "Failed to decode",
    "timeout": "Processing timeout",
    "worker_error": "Processing worker error",
}


def parse_content(content: bytes, provided_type: Optional[str] = None) -> dict:
    content_type = provided_type or detect_mime(content)
    result = {
        "text": None,
        "content_type": content_type,
        "encoding": None,
        "parse_status": None,
    }
    if not content:
        result["parse_status"] = "failed_to_decode"
        return result

    if content_type == "application/pdf":
        text = _extract_pdf_pdfium(content) or _extract_pdf_pymupdf(content) or _extract_pdf_ocr(content)
        result.update(text=text, encoding="utf-8" if text else None,
                      parse_status="decoded" if text else "pdf_extraction_failed")
        return result
    if content_type == "application/vnd.openxmlformats-officedocument.wordprocessingml.document":
        text = _extract_docx(content)
        result.update(text=text, encoding="utf-8" if text else None,
                      parse_status="decoded" if text else "docx_extraction_failed")
        return result
    if content_type in {
        "application/vnd.ms-excel",
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    }:
        text = _extract_excel(content)
        result.update(text=text, encoding="utf-8" if text else None,
                      parse_status="decoded" if text else "excel_extraction_failed")
        return result
    if content_type in {
        "application/vnd.ms-powerpoint",
        "application/vnd.openxmlformats-officedocument.presentationml.presentation",
    }:
        result["parse_status"] = "unsupported_powerpoint"
        return result
    if content_type in {"text/html", "text/xml", "application/xhtml+xml", "application/xml"}:
        text = _extract_html(content)
        result.update(text=text, encoding="utf-8" if text else None,
                      parse_status="decoded" if text else "html_extraction_failed")
        return result
    if content_type in {"text/rtf", "application/rtf"} or content.startswith(RTF_MAGIC_LOOSE):
        text = _extract_rtf(content)
        result.update(text=text, encoding="utf-8" if text else None,
                      parse_status="decoded" if text else "rtf_extraction_failed")
        return result
    if is_binary_content_type(content_type):
        result["parse_status"] = "binary_content" if content_type.startswith(("image/", "audio/", "video/", "font/")) else "unsupported_binary"
        return result

    text, encoding = _guess_text(content)
    result.update(
        text=text,
        encoding=encoding,
        parse_status="decoded" if text else "binary_unknown",
    )
    return result


# ==================== POST PROCESSING ====================

_FTFY_CONFIG = _FtfyConfig(
    fix_c1_controls=True,
    restore_byte_a0=True,
    fix_line_breaks=True,
    normalization="NFC",
    fix_latin_ligatures=False,
)
_GARBAGE_REGEX = re.compile(
    r"f{8,}|(f5ebe8){2,}|(bec7c2){2,}|(333333){2,}|(cdcdcd){2,}|(elnlueve){2,}|(7f){4,}",
    re.IGNORECASE,
)
_OCF_TEXT_PATTERN = re.compile(r"\s*ocf_blob\s*", re.IGNORECASE)
_TEMPLATE_PATTERN = re.compile(r"<%[A-Z_]*%>", re.IGNORECASE)


def _detect_truncation(
    text: str,
    content_type: str,
    raw_bytes: bytes,
    decompressor_flag: bool,
) -> Tuple[bool, Optional[str]]:
    if decompressor_flag:
        return True, "blob_length_mismatch"
    if content_type in {"text/rtf", "application/rtf"} and raw_bytes:
        source = raw_bytes.decode("cp1252", errors="ignore")
        if source.count("{") != source.count("}"):
            return True, "rtf_brace_mismatch"
    if content_type == "application/pdf" and raw_bytes and b"%%EOF" not in raw_bytes[-1024:]:
        return True, "pdf_no_eof"
    return False, None


def post_process(
    text: str,
    content_type: str,
    raw_bytes: bytes,
    decompressor_flag: bool = False,
) -> Tuple[Optional[str], dict]:
    metadata = {
        "ftfy_explain": None,
        "ftfy_failed": False,
        "truncation_flag": False,
        "truncation_reason": None,
    }
    if not isinstance(text, str) or not text:
        return None, metadata

    text = _OCF_TEXT_PATTERN.sub("", text).strip()
    text = _TEMPLATE_PATTERN.sub("", text).strip()
    if not text:
        return None, metadata

    match = _GARBAGE_REGEX.search(text)
    if match:
        if match.start() < 20:
            return None, metadata
        text = text[:match.start()].rstrip()
        metadata["truncation_flag"] = True
        metadata["truncation_reason"] = "garbage_pattern"

    try:
        fixed, explanation = _ftfy.fix_and_explain(text, config=_FTFY_CONFIG)
        text = fixed
        if explanation:
            metadata["ftfy_explain"] = json.dumps([
                (step.transformation, step.encoding)
                if hasattr(step, "transformation") else str(step)
                for step in explanation
            ])
    except Exception:
        metadata["ftfy_failed"] = True

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if not text:
        return None, metadata
    if len(text) > MAX_TEXT_CHARS:
        text = text[:MAX_TEXT_CHARS]
        metadata["truncation_flag"] = True
        metadata["truncation_reason"] = "text_length_cap"

    if not metadata["truncation_flag"]:
        flag, reason = _detect_truncation(
            text, content_type, raw_bytes, decompressor_flag
        )
        metadata["truncation_flag"] = flag
        metadata["truncation_reason"] = reason
    return text, metadata


# ==================== ISOLATED END-TO-END PROCESSING ====================

def process_blob_bytes(raw: bytes, compression_cd, chunk_count: int, blob_length) -> dict:
    metrics = {"chunk_count": int(chunk_count or 0)}
    output = {
        "text": None,
        "content_type": None,
        "encoding": None,
        "binary_size": None,
        "text_length": None,
        "status": None,
        "parse_status": None,
        "truncation_flag": False,
        "truncation_reason": None,
        "ftfy_explain": None,
        "decompression_strategy": None,
        "metrics": metrics,
    }
    decompressed, error = decompress(
        raw, compression_cd, chunk_count, blob_length, metrics,
        max_output_bytes=MAX_DECOMPRESSED_BYTES,
    )
    output["decompression_strategy"] = metrics.get("decompression_strategy")
    if decompressed is None:
        output["status"] = error or "Decompression failed"
        output["parse_status"] = "decompression_failed"
        return output

    output["binary_size"] = len(decompressed)
    parsed = parse_content(decompressed)
    output["content_type"] = parsed["content_type"]
    output["encoding"] = parsed["encoding"]
    output["parse_status"] = parsed["parse_status"]
    metrics["parse_status"] = parsed["parse_status"]

    if parsed["parse_status"] != "decoded":
        output["status"] = STATUS_LABELS.get(parsed["parse_status"], "Failed to decode")
        return output

    cleaned, post = post_process(
        parsed["text"], parsed["content_type"], decompressed,
        decompressor_flag=metrics.get("blob_length_mismatch", False),
    )
    if not cleaned:
        output["status"] = STATUS_LABELS["failed_to_decode"]
        output["parse_status"] = "failed_to_decode"
        return output

    cleaned = cleaned.encode("utf-8", errors="replace").decode("utf-8")
    output.update(
        text=cleaned,
        text_length=len(cleaned),
        status=STATUS_LABELS["decoded"],
        truncation_flag=post["truncation_flag"],
        truncation_reason=post["truncation_reason"],
        ftfy_explain=post["ftfy_explain"],
    )
    metrics["ftfy_failed"] = post["ftfy_failed"]
    return output


class BlobProcessPool:
    """One reusable subprocess per Python worker with hard timeout recovery."""

    def __init__(self, timeout: int = PROCESS_TIMEOUT_SEC):
        self.timeout = timeout
        self.pool = None

    def _ensure(self):
        if self.pool is None:
            self.pool = ProcessPoolExecutor(max_workers=1)

    def _hard_teardown(self):
        pool, self.pool = self.pool, None
        if pool is None:
            return
        processes = list(getattr(pool, "_processes", {}).values())
        for process in processes:
            try:
                process.terminate()
            except Exception:
                pass
        for process in processes:
            try:
                process.join(timeout=2)
                if process.is_alive() and hasattr(process, "kill"):
                    process.kill()
            except Exception:
                pass
        try:
            pool.shutdown(wait=False, kill_workers=True)
        except TypeError:
            try:
                pool.shutdown(wait=False)
            except Exception:
                pass
        except Exception:
            pass

    def process(self, raw: bytes, compression_cd, chunk_count: int, blob_length):
        self._ensure()
        try:
            future = self.pool.submit(
                process_blob_bytes, raw, compression_cd, chunk_count, blob_length
            )
            return future.result(timeout=self.timeout)
        except FutureTimeout:
            self._hard_teardown()
            return {
                "text": None, "content_type": None, "encoding": None,
                "binary_size": None, "text_length": None,
                "status": STATUS_LABELS["timeout"], "parse_status": "timeout",
                "truncation_flag": False, "truncation_reason": None,
                "ftfy_explain": None, "decompression_strategy": None,
                "metrics": {"error": "processing timeout"},
            }
        except Exception as exc:
            self._hard_teardown()
            return {
                "text": None, "content_type": None, "encoding": None,
                "binary_size": None, "text_length": None,
                "status": STATUS_LABELS["worker_error"], "parse_status": "worker_error",
                "truncation_flag": False, "truncation_reason": None,
                "ftfy_explain": None, "decompression_strategy": None,
                "metrics": {"error": f"{type(exc).__name__}: {str(exc)[:300]}"},
            }

    def close(self):
        self._hard_teardown()


print(
    "updated blob_shared_lib loaded; dependency report="
    + json.dumps(dependency_report(), sort_keys=True)
)